# Explicit uncomputation with `forget(q)`

Small examples for this fork's **default-dagger implementation**. Run the cells in order using a Python 3.12+ kernel with this repository's `guppylang` and `guppylang-internals` installed (for example, after `uv sync` in the repository root).

`forget(q)` asks the compiler to undo the computation of a temporary qubit and free it in the zero state. It consumes `q`; it does not measure it. No purity annotation is required. The compiler recognises supported basis-preserving producers and reuses Guppy's existing dagger.

This implementation supports scalar, block-local cleanup. Recursive garbage erasure, allocating helper inverses, and cross-block dependencies are not implemented. See the [implementation report](../docs/uncomputation.md) for the precise scope.

In [1]:
from guppylang import guppy
from guppylang.std.builtins import output
from guppylang.std.quantum import cx, h, measure, qubit, x
from guppylang.std.uncomputation import forget


def run_and_check(entry, expected):
    # Synthesis may allocate extra temporary qubits, so allow some headroom.
    shots = (
        entry.emulator(n_qubits=16)
        .statevector_sim()
        .with_seed(42)
        .with_shots(16)
        .run()
    )
    records = [dict(shot) for shot in shots]
    assert len(records) == 16
    assert all(record == expected for record in records), records
    print(f"Verified {len(records)} shots: {expected}")

## 1. Forget a locally computed bit

The qubit starts in zero and X changes it to one. `forget` synthesizes cleanup instead of discarding the one state. This first example only compiles the function; the later examples also execute on the statevector simulator.

In [2]:
@guppy
def forget_one() -> None:
    temporary = qubit()
    x(temporary)
    forget(temporary)


_ = forget_one.compile_function()
print("Compiled: X is undone before the temporary is freed.")

Compiled: X is undone before the temporary is freed.


## 2. Compute, use, forget — through a helper

`compute` is an ordinary reversible Guppy helper. Its control `source` is conserved; its target changes. `copy_via_temporary` uses the computed temporary to update `result`, then requests cleanup.

Preparation and measurement live in the outer function because this baseline rejects measurement in a block containing `forget`. Both outputs should be **True** when the source starts in one.

In [3]:
@guppy(unitary=True)
def compute(source: qubit, target: qubit) -> None:
    cx(source, target)


@guppy
def copy_via_temporary(source: qubit, result: qubit) -> None:
    temporary = qubit()
    compute(source, temporary)
    cx(temporary, result)
    forget(temporary)


@guppy
def copy_one() -> None:
    source = qubit()
    result = qubit()
    x(source)
    copy_via_temporary(source, result)
    output("source", bool(measure(source)))
    output("result", bool(measure(result)))


run_and_check(copy_one, {"source": True, "result": True})

Verified 16 shots: {'source': True, 'result': True}


## 3. Cleanup preserves coherence

Now put `source` in superposition. Copying through the temporary entangles it with the other qubits, so discarding the temporary would generally destroy coherence.

After automatic cleanup, a CX disentangles `result` and a second H returns `source` to zero. Both measurements should be **False** in every shot.

In [4]:
@guppy
def coherent_copy() -> None:
    source = qubit()
    result = qubit()
    h(source)
    copy_via_temporary(source, result)
    cx(source, result)
    h(source)
    output("source", bool(measure(source)))
    output("result", bool(measure(result)))


run_and_check(coherent_copy, {"source": False, "result": False})

Verified 16 shots: {'source': False, 'result': False}


## 4. A forgotten value is still needed for later cleanup

`second` depends on `first`, but the source asks to forget `first` first. The backwards pass extends its lifetime until the inverse for `second` has used it. You do not need to write the inverse calls or choose their order.

The returned `result` remains available. We check the same coherent round trip as above.

In [5]:
@guppy
def dependency_chain(source: qubit) -> qubit:
    first = qubit()
    cx(source, first)
    second = qubit()
    cx(first, second)
    result = qubit()
    cx(second, result)
    forget(first)
    forget(second)
    return result


@guppy
def coherent_chain() -> None:
    source = qubit()
    h(source)
    result = dependency_chain(source)
    cx(source, result)
    h(source)
    output("source", bool(measure(source)))
    output("result", bool(measure(result)))


run_and_check(coherent_chain, {"source": False, "result": False})

Verified 16 shots: {'source': False, 'result': False}


## 5. A helper changes its source before copying

This helper first flips `a`, then copies its **updated** value to the temporary. Cleanup must preserve the updated `a`. Treating the helper as one indivisible call is too coarse: reversing the whole helper would also reverse the X.

When that whole-call analysis fails, the compiler exposes the supported helper body. It can then undo just the temporary's CX computation. This example also checks the HUGR serialization and DOT-generation steps.

In [6]:
@guppy(unitary=True)
def use_cx(a: qubit, b: qubit) -> None:
    x(a)
    cx(a, b)


@guppy
def clean_updated_copy(a: qubit, result: qubit) -> None:
    temporary = qubit()
    use_cx(a, temporary)
    cx(temporary, result)
    forget(temporary)


@guppy
def updated_copy() -> None:
    a = qubit()
    result = qubit()
    clean_updated_copy(a, result)
    output("source", bool(measure(a)))
    output("result", bool(measure(result)))


program = clean_updated_copy.compile_function().modules[0]
serialized = program.to_bytes()
graph = program.render_dot()
assert serialized and graph.source
run_and_check(updated_copy, {"source": True, "result": True})

# To display the graph in another cell, evaluate: graph
# To save the HUGR, use: Path("complex_guppy.hugr").write_bytes(serialized)
# (Import Path from pathlib first.)

Verified 16 shots: {'source': True, 'result': True}


The same cleanup also works when `h(a)` is inside the helper containing `forget`. The superposed `a` survives and remains available as a control; only the temporary is uncomputed. This round trip undoes the retained CX, X, and H effects and checks that both measurements are zero.

In [7]:
@guppy
def clean_updated_superposition(a: qubit, result: qubit) -> None:
    temporary = qubit()
    h(a)
    use_cx(a, temporary)
    cx(temporary, result)
    forget(temporary)


@guppy
def updated_superposition() -> None:
    a = qubit()
    result = qubit()
    clean_updated_superposition(a, result)
    cx(a, result)
    x(a)
    h(a)
    output("source", bool(measure(a)))
    output("result", bool(measure(result)))


program = clean_updated_superposition.compile_function().modules[0]
assert program.to_bytes() and program.render_dot().source
run_and_check(updated_superposition, {"source": False, "result": False})

Verified 16 shots: {'source': False, 'result': False}


## 6. An expected error: forgetting a superposition

H is controllable and daggerable, but it is not a basis-preserving producer. The current `forget` contract rejects this request. This differs from example 3: there, the temporary was computed by CX from a **still-available** superposed source.

The error is a Python `RuntimeError` raised **during compilation**, not an error emitted onto the quantum device. We catch it here so **Run All** succeeds.

In [8]:
@guppy
def invalid_forget() -> None:
    temporary = qubit()
    h(temporary)
    forget(temporary)


try:
    invalid_forget.compile_function()
except RuntimeError as error:
    assert "computational basis" in str(error)
    print("Expected RuntimeError:", error)
else:
    raise AssertionError("Expected this uncomputation request to be rejected")

Expected RuntimeError: Cannot forget register 1: h is not known to preserve the computational basis without phases


## Try changing an example

- Remove `x(source)` in example 2: change the expected outputs to both `False`.
- In `copy_via_temporary`, insert `x(source)` just before `forget(temporary)`: compilation should fail because cleanup needs the old source value.
- Keep preparation, measurement, and any surrounding control flow outside the helper containing `forget`.

For algorithm details, supported operations, and remaining limitations, see [the implementation report](../docs/uncomputation.md).